# Benchmarking against ViralForsight

## Create the groups

In [ ]:
validation_source_sequences = 'Fasta file containing the validation source sequences'
validation_predictions_prefix_swap_file = 'Fasta file containing the validation predictions with prefix swap'
VF_unranked_sequences_path = 'Path to the VF generated sequences'
VF_ranked_sequences_path = 'Path to the VF ranked sequences'

In [ ]:
import os

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[line_category] = sequence
                else:
                    first = False
                line_category = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[line_category] = sequence
    return sequence_types

val_sequences = read_fasta(validation_source_sequences)
all_preds = read_fasta(validation_predictions_prefix_swap_file)

R_groups = {}
current_group = 0
for accession in list(val_sequences.keys()):
    current_group_seqs = {}
    for pred, seq in all_preds.items():
        if accession in pred:
            current_group = pred.split('row=')[1].split(' ')[0]
            if current_group not in current_group_seqs:
                current_group_seqs[current_group] = []
            current_group_seqs[current_group].append(seq)
    R_groups[accession] = current_group_seqs

VF_groups = {}
for accession in list(val_sequences.keys()):
    file_name = os.path.join(VF_unranked_sequences_path, accession, f'{accession}_raw_seqs')
    VF_groups[accession] = {}
    sequences = []
    for ix,file in enumerate(os.listdir(file_name)):
        with open(os.path.join(file_name, file), 'r') as f:
            for line in f.readlines():
                sequences.append(line.strip())
    VF_groups[accession]['unranked'] = list(set(sequences))

rbds_expr = {}
rbds_bind = {}
filtered_s1_sequences = {}

for accession in list(val_sequences.keys()):
    rbd_expr_path = os.path.join(VF_ranked_sequences_path, accession, 'top_20_candidates_expr.fasta')
    rbd_bind_path = os.path.join(VF_ranked_sequences_path, accession, 'top_20_candidates_bind.fasta')
    rbds_expr[accession] = read_fasta(rbd_expr_path)
    rbds_bind[accession] = read_fasta(rbd_bind_path)

    generated_s1_sequences_path = os.path.join(VF_ranked_sequences_path, accession, f'{accession}_raw_seqs')
    filtered_s1_sequences[accession] = []
    for file in os.listdir(generated_s1_sequences_path):
        s1_sequences = []
        with open(os.path.join(generated_s1_sequences_path, file), 'r') as f:
            for line in f.readlines():
                s1_sequences.append(line.strip())
        for seq in s1_sequences:
            for rbd in set(list(rbds_expr[accession].values()) + list(rbds_bind[accession].values())):
                if rbd in seq:
                    filtered_s1_sequences[accession].append(seq)
                    break

for accession in list(filtered_s1_sequences.keys()):
    VF_groups[accession]['bind'] = []
    VF_groups[accession]['expr'] = []

    for seq in filtered_s1_sequences[accession]:
        for rbd in rbds_bind[accession].values():
            if rbd in seq:
                VF_groups[accession]['bind'].append(seq)
    for seq in filtered_s1_sequences[accession]:
        for rbd in rbds_expr[accession].values():
            if rbd in seq:
                VF_groups[accession]['expr'].append(seq)


## Compute the distances

In [ ]:
validation_source_sequences = 'Fasta file containing the validation source sequences'
validation_ground_truth_file = 'Pickle file containing the ground truth sequences'
all_sequences_fasta = 'Fasta file containing all the sequences'

In [ ]:
import pickle as pkl
import numpy as np
from Bio.Align import substitution_matrices

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[line_category] = sequence
                else:
                    first = False
                line_category = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[line_category] = sequence
    return sequence_types

def compute_pairwise_blosum_distances(sequences, matrix_name='BLOSUM62', gap_penalty=None):
    similarity_matrix = compute_pairwise_blosum_similarity(sequences, matrix_name, gap_penalty)
    return translate_similarity_to_distance(similarity_matrix)

def compute_pairwise_blosum_similarity(sequences, matrix_name, gap_penalty):
    matrix = substitution_matrices.load(matrix_name)
    matrix_vals = np.array(matrix.values()).reshape(24, 24)
    if gap_penalty is not None:
        max_penalty = gap_penalty
    else:
        max_penalty = np.min(matrix_vals) - 1
    lookup = np.ones((len(matrix_vals) + 1, len(matrix_vals) + 1), dtype=np.float64) * max_penalty
    lookup[-1, -1] = 0
    lookup[:-1, :-1] = matrix_vals
    alphabet = matrix.alphabet + '-'
    aa_to_idx = {aa: i for i, aa in enumerate(alphabet)}
    seqs_int = np.array(
        [[aa_to_idx[aa.upper()] for aa in seq] for seq in sequences],
        dtype=np.uint8,
    )
    n_sequences, length = seqs_int.shape
    pw_sims = np.zeros((n_sequences, n_sequences), dtype=np.float64)
    for position in range(length):
        col = seqs_int[:, position]
        pw_sims += lookup[col[:, None], col]
    return pw_sims

def translate_similarity_to_distance(similarity_matrix):
    diag = np.diag(similarity_matrix)
    dist_sq = diag[:, None] + diag[None, :] - 2.0 * similarity_matrix
    np.maximum(dist_sq, 0.0, out=dist_sq)
    return np.sqrt(dist_sq)

val_sequences = read_fasta(validation_source_sequences)

gt_sequences = pkl.load(open(validation_ground_truth_file, 'rb'))

sample_gt = str(next(iter(gt_sequences.values()))[0])

msa_raw = read_fasta(all_sequences_fasta)
msa_by_id = {
    k.split('genome_id:')[1].split('|')[0]: v
    for k, v in msa_raw.items() if 'genome_id:' in k
}
gt_sequences = {
    acc: [msa_by_id[str(gt_id)] for gt_id in gt_ids]
    for acc, gt_ids in gt_sequences.items()
}

all_sequences = {}
for ix,accession in enumerate(list(val_sequences.keys())):
    sequences = {}
    sequences['source'] = [val_sequences[accession]]
    sequences['gt'] = gt_sequences[accession]
    for group in R_groups[accession].keys():
        sequences[f'R_{group}'] = R_groups[accession][group]
    for group in VF_groups[accession].keys():
        sequences[f'VF_{group}'] = VF_groups[accession][group]
    all_sequences[accession] = sequences

### Align the length-mismatched sequences

In [ ]:
aligned_sequences_path = 'Sequences where to store and align the faulty sequences/'

In [ ]:
import os

aligned_sequences_path = '/Users/nle68017/code/NLEProgress/data/VF_spike/aligned_preds/'
os.makedirs(aligned_sequences_path, exist_ok=True)

with open(os.path.join(aligned_sequences_path, 'base_seqs.fasta'), 'w') as f:
    with open(os.path.join(aligned_sequences_path, 'to_align.fasta'), 'w') as f2:
        for accession in all_sequences.keys():
            for group in all_sequences[accession].keys():
                for seq in all_sequences[accession][group]:
                    N = len(all_sequences[accession]['source'][0])
                    if len(seq) != len(all_sequences[accession]['source'][0]):
                        f2.write(f'>{accession}_{group}_{n}\n{seq.replace("-", "")}\n')
                    else:
                        f.write(f'>{accession}_{group}_{n}\n{seq}\n')
                    n += 1

In [ ]:
!mafft --add {aligned_sequences_path}to_align.fasta {aligned_sequences_path}base_seqs.fasta > {aligned_sequences_path}all_aligned.fasta

In [ ]:
predicted_sequences_aligned = {}
aligned_sequences_path = os.path.join(aligned_sequences_path, 'all_aligned.fasta')
aligned_sequences = read_fasta(aligned_sequences_path)
for label, seq in aligned_sequences.items():
    accession = label.split('_')[0]
    group = label.split('_')[1:-1]
    group = '_'.join(group)
    if accession not in predicted_sequences_aligned:
        predicted_sequences_aligned[accession] = {}
    if group not in predicted_sequences_aligned[accession]:
        predicted_sequences_aligned[accession][group] = []
    predicted_sequences_aligned[accession][group].append(seq)

all_sequences = predicted_sequences_aligned

### Continue with the distance calculation

In [ ]:
pw_distances_by_group = {}

for accession in all_sequences.keys():
    all_seqs = [seq for group in all_sequences[accession].keys() for seq in all_sequences[accession][group]]

    D = compute_pairwise_blosum_distances(all_seqs)

    group_starts = {}
    added_seqs = 0
    for group in all_sequences[accession].keys():
        group_starts[group] = added_seqs
        added_seqs += len(all_sequences[accession][group])
    gt_start = group_starts['gt']
    gt_end = gt_start + len(all_sequences[accession]['gt'])
    gt_rows = D[gt_start:gt_end]
    pw_distances_groups = {}
    for group in all_sequences[accession].keys():
        start = group_starts[group]
        end = start + len(all_sequences[accession][group])
        pw_distances_groups[group] = gt_rows[:, start:end]
    
    pw_distances_by_group[accession] = pw_distances_groups

## Violinplots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

accession_to_lineage = {'2697049.7941247': 'BA.5.2.1', '2697049.6371208': 'BA.5.2.1', '2697049.8105664': 'BA.5.2', '2697049.6795617': 'BA.5.2', '2697049.6675091': 'BA.5.2.1', '2697049.6705819': 'BA.5.2.1', '2697049.6591624': 'BA.5.1', '2697049.6784537': 'BA.5.2', '2697049.6285062': 'BA.5.1', '2697049.8784361': 'BA.5.2', '2697049.5958919': 'BA.5.2.1', '2697049.7089975': 'BA.5.2', '2697049.6916394': 'BA.5.1', '2697049.6045006': 'BA.5.1', '2697049.4573707': 'BA.1.1', '2697049.8125292': 'BA.5.1'}

GROUP_COLORS = {
    'source': '#d62728',       # red
    'R': '#1f77b4',            # blue
    'VF_unranked': '#ff7f0e',  # orange
    'VF_bind': '#2ca02c',      # green
    'VF_expr': '#9467bd',      # purple
}
GT_MARKERS = ['o', '^', 's', 'D', 'v', 'P', 'X']

def get_group_color(group):
    if group == 'source':
        return GROUP_COLORS['source']
    elif group.startswith('R_'):
        return GROUP_COLORS['R']
    elif group in GROUP_COLORS:
        return GROUP_COLORS[group]
    return '#7f7f7f'

def format_group_label(group):
    if group.startswith('R_'):
        return f'row_{group[2:]}'
    return group

def plot_blosum_distances(pw_distances_by_group):
    accessions = list(pw_distances_by_group.keys())
    n_rows, n_cols = 4, 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 22))

    for idx, accession in enumerate(accessions):
        ax = axes[idx // n_cols, idx % n_cols]
        groups = [g for g in pw_distances_by_group[accession].keys() if g != 'gt']
        xtick_labels = [format_group_label(g) for g in groups]

        # Keep raw (n_gt, n_seqs) arrays per group and pooled 1D arrays for violin/boxplot
        group_raw = [np.atleast_2d(np.asarray(pw_distances_by_group[accession][g])) for g in groups]
        group_dists = [r.ravel() for r in group_raw]
        group_mins = [np.min(d) if len(d) > 0 else np.inf for d in group_dists]
        overall_min = np.min(group_mins) if len(group_mins) > 0 else np.inf
        winning_indices = [
            i for i, m in enumerate(group_mins)
            if np.isfinite(m) and np.isclose(m, overall_min, rtol=0.0, atol=1e-9)
        ]

        non_empty = [d for d in group_dists if len(d) > 0]
        all_vals = np.concatenate(non_empty) if non_empty else np.array([0.0])
        y_span = max(np.ptp(all_vals), 1.0)
        y_pad = 0.04 * y_span
        n_gts = group_raw[0].shape[0] if len(group_raw) > 0 else 1

        for i, (group, raw, dists) in enumerate(zip(groups, group_raw, group_dists)):
            if len(dists) == 0:
                continue
            color = get_group_color(group)

            if len(dists) >= 2 and np.ptp(dists) > 0:
                parts = ax.violinplot(
                    [dists],
                    positions=[i],
                    widths=0.6,
                    showmeans=False,
                    showmedians=True,
                    showextrema=True,
                )
                for pc in parts['bodies']:
                    pc.set_facecolor(color)
                    pc.set_edgecolor('black')
                    pc.set_alpha(0.45)
                    pc.set_zorder(2)
                for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
                    if partname in parts:
                        parts[partname].set_edgecolor('black' if partname == 'cmedians' else color)
                        parts[partname].set_linewidth(1.2)
                        parts[partname].set_zorder(2)
            else:
                ax.hlines(dists[0], i - 0.25, i + 0.25, color=color, linewidth=2.0, zorder=2)

            for gt_idx in range(raw.shape[0]):
                gt_dists = raw[gt_idx]
                if len(gt_dists) == 0:
                    continue
                rng = np.random.default_rng(42 + idx * 1000 + i * 10 + gt_idx)
                jitter = (
                    rng.uniform(-0.12, 0.12, size=len(gt_dists))
                    if (len(gt_dists) > 1 or raw.shape[0] > 1)
                    else np.zeros(len(gt_dists))
                )
                marker = GT_MARKERS[gt_idx % len(GT_MARKERS)]
                lbl = f'gt_{gt_idx + 1}' if (i == 0 and n_gts > 1) else '_nolegend_'
                ax.scatter(
                    np.full(len(gt_dists), i) + jitter,
                    gt_dists,
                    color=color,
                    marker=marker,
                    s=24 if n_gts > 1 else 20,
                    alpha=0.8,
                    edgecolors='black',
                    linewidths=0.4,
                    label=lbl,
                    zorder=3,
                )

        if n_gts > 1:
            leg = ax.legend(title='Ground truth', fontsize='x-small', title_fontsize='x-small', loc='upper left')
            for handle in leg.legend_handles:
                handle.set_facecolor('gray')

        for win_i in winning_indices:
            dists = group_dists[win_i]
            y_min, y_max = np.min(dists), np.max(dists)
            rect = Rectangle(
                (win_i - 0.42, y_min - y_pad),
                0.84,
                (y_max - y_min) + 2 * y_pad,
                fill=False,
                edgecolor='black',
                linewidth=1.8,
                zorder=4,
            )
            ax.add_patch(rect)

        ax.set_title(f'{accession} ({accession_to_lineage[accession]})')
        ax.set_xlim(-0.6, len(groups) - 0.4)
        ax.set_ylim(bottom=min(-0.5, np.min(all_vals) - 2 * y_pad), top=np.max(all_vals) + 2.5 * y_pad)
        ax.set_xticks(range(len(groups)))
        tick_labels = ax.set_xticklabels(xtick_labels, rotation=45, ha='right')
        for win_i in winning_indices:
            tick_labels[win_i].set_fontweight('bold')

        if idx % n_cols == 0:
            ax.set_ylabel('BLOSUM distance to gt')
        else:
            ax.set_ylabel('')
        ax.set_xlabel('Group')

    for idx in range(len(accessions), n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].axis('off')

    legend_handles = [
        Patch(facecolor=GROUP_COLORS['source'], edgecolor='black', alpha=0.6, label='source'),
        Patch(facecolor=GROUP_COLORS['R'], edgecolor='black', alpha=0.6, label='row_* (R_*)'),
        Patch(facecolor=GROUP_COLORS['VF_unranked'], edgecolor='black', alpha=0.6, label='VF_unranked'),
        Patch(facecolor=GROUP_COLORS['VF_bind'], edgecolor='black', alpha=0.6, label='VF_bind'),
        Patch(facecolor=GROUP_COLORS['VF_expr'], edgecolor='black', alpha=0.6, label='VF_expr'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.965), fontsize=12)
    fig.suptitle('Pairwise BLOSUM distances to ground truth per group (Violin plots)', fontsize=18, y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_blosum_distances(pw_distances_by_group)

## RBD

In [ ]:
rbd_start = 357
rbd_end = 558

In [ ]:
pw_distances_by_group_rbd = {}
for accession in all_sequences.keys():
    all_seqs = [seq[rbd_start:rbd_end] for group in all_sequences[accession].keys() for seq in all_sequences[accession][group]]
    D = compute_pairwise_blosum_distances(all_seqs)
    group_starts = {}
    added_seqs = 0
    for group in all_sequences[accession].keys():
        group_starts[group] = added_seqs
        added_seqs += len(all_sequences[accession][group])
    gt_start = group_starts['gt']
    gt_end = gt_start + len(all_sequences[accession]['gt'])
    gt_rows = D[gt_start:gt_end]
    pw_distances_groups = {}
    for group in all_sequences[accession].keys():
        start = group_starts[group]
        end = start + len(all_sequences[accession][group])
        pw_distances_groups[group] = gt_rows[:, start:end]
    
    pw_distances_by_group_rbd[accession] = pw_distances_groups

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

accession_to_lineage = {'2697049.7941247': 'BA.5.2.1', '2697049.6371208': 'BA.5.2.1', '2697049.8105664': 'BA.5.2', '2697049.6795617': 'BA.5.2', '2697049.6675091': 'BA.5.2.1', '2697049.6705819': 'BA.5.2.1', '2697049.6591624': 'BA.5.1', '2697049.6784537': 'BA.5.2', '2697049.6285062': 'BA.5.1', '2697049.8784361': 'BA.5.2', '2697049.5958919': 'BA.5.2.1', '2697049.7089975': 'BA.5.2', '2697049.6916394': 'BA.5.1', '2697049.6045006': 'BA.5.1', '2697049.4573707': 'BA.1.1', '2697049.8125292': 'BA.5.1'}

GROUP_COLORS = {
    'source': '#d62728',       # red
    'R': '#1f77b4',            # blue
    'VF_unranked': '#ff7f0e',  # orange
    'VF_bind': '#2ca02c',      # green
    'VF_expr': '#9467bd',      # purple
}
GT_MARKERS = ['o', '^', 's', 'D', 'v', 'P', 'X']

def get_group_color(group):
    if group == 'source':
        return GROUP_COLORS['source']
    elif group.startswith('R_'):
        return GROUP_COLORS['R']
    elif group in GROUP_COLORS:
        return GROUP_COLORS[group]
    return '#7f7f7f'

def format_group_label(group):
    if group.startswith('R_'):
        return f'row_{group[2:]}'
    return group

def plot_blosum_distances(pw_distances_by_group):
    accessions = list(pw_distances_by_group.keys())
    n_rows, n_cols = 4, 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 22))

    for idx, accession in enumerate(accessions):
        ax = axes[idx // n_cols, idx % n_cols]
        groups = [g for g in pw_distances_by_group[accession].keys() if g != 'gt']
        xtick_labels = [format_group_label(g) for g in groups]

        # Keep raw (n_gt, n_seqs) arrays per group and pooled 1D arrays for violin/boxplot
        group_raw = [np.atleast_2d(np.asarray(pw_distances_by_group[accession][g])) for g in groups]
        group_dists = [r.ravel() for r in group_raw]
        group_mins = [np.min(d) if len(d) > 0 else np.inf for d in group_dists]
        overall_min = np.min(group_mins) if len(group_mins) > 0 else np.inf
        winning_indices = [
            i for i, m in enumerate(group_mins)
            if np.isfinite(m) and np.isclose(m, overall_min, rtol=0.0, atol=1e-9)
        ]

        non_empty = [d for d in group_dists if len(d) > 0]
        all_vals = np.concatenate(non_empty) if non_empty else np.array([0.0])
        y_span = max(np.ptp(all_vals), 1.0)
        y_pad = 0.04 * y_span
        n_gts = group_raw[0].shape[0] if len(group_raw) > 0 else 1

        for i, (group, raw, dists) in enumerate(zip(groups, group_raw, group_dists)):
            if len(dists) == 0:
                continue
            color = get_group_color(group)

            if len(dists) >= 2 and np.ptp(dists) > 0:
                parts = ax.violinplot(
                    [dists],
                    positions=[i],
                    widths=0.6,
                    showmeans=False,
                    showmedians=True,
                    showextrema=True,
                )
                for pc in parts['bodies']:
                    pc.set_facecolor(color)
                    pc.set_edgecolor('black')
                    pc.set_alpha(0.45)
                    pc.set_zorder(2)
                for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
                    if partname in parts:
                        parts[partname].set_edgecolor('black' if partname == 'cmedians' else color)
                        parts[partname].set_linewidth(1.2)
                        parts[partname].set_zorder(2)
            else:
                ax.hlines(dists[0], i - 0.25, i + 0.25, color=color, linewidth=2.0, zorder=2)

            for gt_idx in range(raw.shape[0]):
                gt_dists = raw[gt_idx]
                if len(gt_dists) == 0:
                    continue
                rng = np.random.default_rng(42 + idx * 1000 + i * 10 + gt_idx)
                jitter = (
                    rng.uniform(-0.12, 0.12, size=len(gt_dists))
                    if (len(gt_dists) > 1 or raw.shape[0] > 1)
                    else np.zeros(len(gt_dists))
                )
                marker = GT_MARKERS[gt_idx % len(GT_MARKERS)]
                lbl = f'gt_{gt_idx + 1}' if (i == 0 and n_gts > 1) else '_nolegend_'
                ax.scatter(
                    np.full(len(gt_dists), i) + jitter,
                    gt_dists,
                    color=color,
                    marker=marker,
                    s=24 if n_gts > 1 else 20,
                    alpha=0.8,
                    edgecolors='black',
                    linewidths=0.4,
                    label=lbl,
                    zorder=3,
                )

        if n_gts > 1:
            leg = ax.legend(title='Ground truth', fontsize='x-small', title_fontsize='x-small', loc='upper left')
            for handle in leg.legend_handles:
                handle.set_facecolor('gray')

        # Surround winning group(s) with a box (handling ties)
        for win_i in winning_indices:
            dists = group_dists[win_i]
            y_min, y_max = np.min(dists), np.max(dists)
            rect = Rectangle(
                (win_i - 0.42, y_min - y_pad),
                0.84,
                (y_max - y_min) + 2 * y_pad,
                fill=False,
                edgecolor='black',
                linewidth=1.8,
                zorder=4,
            )
            ax.add_patch(rect)

        ax.set_title(f'{accession} ({accession_to_lineage[accession]})')
        ax.set_xlim(-0.6, len(groups) - 0.4)
        ax.set_ylim(bottom=min(-0.5, np.min(all_vals) - 2 * y_pad), top=np.max(all_vals) + 2.5 * y_pad)
        ax.set_xticks(range(len(groups)))
        tick_labels = ax.set_xticklabels(xtick_labels, rotation=45, ha='right')
        for win_i in winning_indices:
            tick_labels[win_i].set_fontweight('bold')

        if idx % n_cols == 0:
            ax.set_ylabel('BLOSUM distance to gt')
        else:
            ax.set_ylabel('')
        ax.set_xlabel('Group')

    for idx in range(len(accessions), n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].axis('off')

    legend_handles = [
        Patch(facecolor=GROUP_COLORS['source'], edgecolor='black', alpha=0.6, label='source'),
        Patch(facecolor=GROUP_COLORS['R'], edgecolor='black', alpha=0.6, label='row_* (R_*)'),
        Patch(facecolor=GROUP_COLORS['VF_unranked'], edgecolor='black', alpha=0.6, label='VF_unranked'),
        Patch(facecolor=GROUP_COLORS['VF_bind'], edgecolor='black', alpha=0.6, label='VF_bind'),
        Patch(facecolor=GROUP_COLORS['VF_expr'], edgecolor='black', alpha=0.6, label='VF_expr'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.965), fontsize=12)
    fig.suptitle(f'Pairwise BLOSUM distances to ground truth per group (Violin plots)', fontsize=18, y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_blosum_distances(pw_distances_by_group_rbd)

## Summary tables

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

def plot_mean_std_and_min_summary_table(pw_dist_dict, title_suffix='S1'):
    col_defs = [
        ('source',      'd(source)'),
        ('R',           'R (row_*)'),
        ('VF_unranked', 'VF_unranked'),
        ('VF_bind',     'VF_bind'),
        ('VF_expr',     'VF_expr'),
        ('VF_comb',     'VF_combined'),
    ]

    row_labels = []
    means_mat = []
    stds_mat = []
    mins_mat = []
    mins_stds_mat = []

    all_R, all_VF = [], []
    all_VF_sub = {'VF_unranked': [], 'VF_bind': [], 'VF_expr': []}
    all_src = []

    best_src_per_acc = []
    best_R_per_acc = []
    best_VF_per_acc = []
    best_VF_sub_per_acc = {'VF_unranked': [], 'VF_bind': [], 'VF_expr': []}

    for accession in pw_dist_dict.keys():
        groups = [
            g for g in pw_dist_dict[accession].keys()
            if g != 'gt' and np.asarray(pw_dist_dict[accession][g]).size > 0
        ]
        n_gts = np.atleast_2d(pw_dist_dict[accession][groups[0]]).shape[0]

        distances_VF, distances_R = [], []
        sub_dists = {'VF_unranked': [], 'VF_bind': [], 'VF_expr': []}
        for group in groups:
            flat_dists = np.asarray(pw_dist_dict[accession][group]).ravel()
            if group.startswith('R_'):
                distances_R.extend(flat_dists)
            elif group.startswith('VF_'):
                distances_VF.extend(flat_dists)
                if group in sub_dists:
                    sub_dists[group].extend(flat_dists)
                    all_VF_sub[group].extend(flat_dists)

        src_flat = np.asarray(pw_dist_dict[accession]['source']).ravel()
        all_src.extend(src_flat)
        all_R.extend(distances_R)
        all_VF.extend(distances_VF)

        lineage = accession_to_lineage.get(accession, '')
        gt_tag = 'agg (4 GTs)' if n_gts > 1 else '1 GT'
        row_labels.append(f'{accession} ({lineage}) [{gt_tag}]')
        means_mat.append([
            np.mean(src_flat),
            np.mean(distances_R),
            np.mean(sub_dists['VF_unranked']),
            np.mean(sub_dists['VF_bind']),
            np.mean(sub_dists['VF_expr']),
            np.mean(distances_VF),
        ])
        stds_mat.append([
            np.std(src_flat),
            np.std(distances_R),
            np.std(sub_dists['VF_unranked']),
            np.std(sub_dists['VF_bind']),
            np.std(sub_dists['VF_expr']),
            np.std(distances_VF),
        ])

        row_mins = [
            np.min(src_flat),
            np.min(distances_R),
            np.min(sub_dists['VF_unranked']),
            np.min(sub_dists['VF_bind']),
            np.min(sub_dists['VF_expr']),
            np.min(distances_VF),
        ]
        mins_mat.append(row_mins)
        mins_stds_mat.append([0.0] * 6)

        best_src_per_acc.append(row_mins[0])
        best_R_per_acc.append(row_mins[1])
        best_VF_sub_per_acc['VF_unranked'].append(row_mins[2])
        best_VF_sub_per_acc['VF_bind'].append(row_mins[3])
        best_VF_sub_per_acc['VF_expr'].append(row_mins[4])
        best_VF_per_acc.append(row_mins[5])

        if n_gts > 1:
            for gt_idx in range(n_gts):
                gt_VF = [d for g in groups if g.startswith('VF_') for d in np.atleast_2d(pw_dist_dict[accession][g])[gt_idx]]
                gt_R = [d for g in groups if g.startswith('R_') for d in np.atleast_2d(pw_dist_dict[accession][g])[gt_idx]]
                gt_sub = {
                    sub: np.atleast_2d(pw_dist_dict[accession][sub])[gt_idx]
                    for sub in ('VF_unranked', 'VF_bind', 'VF_expr')
                }
                src_gt = np.atleast_2d(pw_dist_dict[accession]['source'])[gt_idx]
                row_labels.append(f'    ↳ gt_{gt_idx + 1}')
                means_mat.append([
                    np.mean(src_gt),
                    np.mean(gt_R),
                    np.mean(gt_sub['VF_unranked']),
                    np.mean(gt_sub['VF_bind']),
                    np.mean(gt_sub['VF_expr']),
                    np.mean(gt_VF),
                ])
                stds_mat.append([
                    0.0,
                    np.std(gt_R),
                    np.std(gt_sub['VF_unranked']),
                    np.std(gt_sub['VF_bind']),
                    np.std(gt_sub['VF_expr']),
                    np.std(gt_VF),
                ])
                mins_mat.append([
                    np.min(src_gt),
                    np.min(gt_R),
                    np.min(gt_sub['VF_unranked']),
                    np.min(gt_sub['VF_bind']),
                    np.min(gt_sub['VF_expr']),
                    np.min(gt_VF),
                ])
                mins_stds_mat.append([0.0] * 6)

    # Add overall summary row at the bottom
    row_labels.append('Overall (all accessions)')
    means_mat.append([
        np.mean(all_src),
        np.mean(all_R),
        np.mean(all_VF_sub['VF_unranked']),
        np.mean(all_VF_sub['VF_bind']),
        np.mean(all_VF_sub['VF_expr']),
        np.mean(all_VF),
    ])
    stds_mat.append([
        np.std(all_src),
        np.std(all_R),
        np.std(all_VF_sub['VF_unranked']),
        np.std(all_VF_sub['VF_bind']),
        np.std(all_VF_sub['VF_expr']),
        np.std(all_VF),
    ])
    mins_mat.append([
        np.mean(best_src_per_acc),
        np.mean(best_R_per_acc),
        np.mean(best_VF_sub_per_acc['VF_unranked']),
        np.mean(best_VF_sub_per_acc['VF_bind']),
        np.mean(best_VF_sub_per_acc['VF_expr']),
        np.mean(best_VF_per_acc),
    ])
    mins_stds_mat.append([
        np.std(best_src_per_acc),
        np.std(best_R_per_acc),
        np.std(best_VF_sub_per_acc['VF_unranked']),
        np.std(best_VF_sub_per_acc['VF_bind']),
        np.std(best_VF_sub_per_acc['VF_expr']),
        np.std(best_VF_per_acc),
    ])

    means_mat = np.asarray(means_mat)
    stds_mat = np.asarray(stds_mat)
    mins_mat = np.asarray(mins_mat)
    mins_stds_mat = np.asarray(mins_stds_mat)
    n_rows, n_cols = means_mat.shape

    vmin = min(np.nanmin(means_mat), np.nanmin(mins_mat))
    vmax = max(np.nanmax(means_mat), np.nanmax(mins_mat))
    thresh = vmin + 0.55 * (vmax - vmin)

    fig, (ax_mean, ax_min) = plt.subplots(
        1, 2, figsize=(18, 0.52 * n_rows + 2.4),
        gridspec_kw={'width_ratios': [1.0, 1.0], 'wspace': 0.12},
        sharey=True,
    )

    def draw_heatmap_block(ax, val_mat, std_mat, block_title, show_std_all_rows=True):
        im = ax.imshow(val_mat, cmap='YlGnBu', aspect='auto', vmin=vmin, vmax=vmax)
        for r in range(n_rows):
            row_min = np.nanmin(val_mat[r])
            for c in range(n_cols):
                v, s = val_mat[r, c], std_mat[r, c]
                is_best = np.isclose(v, row_min, rtol=0.0, atol=1e-6)
                if show_std_all_rows:
                    txt = f'{v:.2f}' if (c == 0 and s == 0) else f'{v:.2f} ± {s:.2f}'
                else:
                    txt = f'{v:.2f} ± {s:.2f}' if (r == n_rows - 1 and s > 0) else f'{v:.2f}'
                text_color = 'white' if v > thresh else 'black'
                ax.text(
                    c, r, txt,
                    ha='center', va='center',
                    fontsize=8.5,
                    fontweight='bold' if is_best else 'normal',
                    color=text_color,
                )
                if is_best:
                    ax.add_patch(Rectangle(
                        (c - 0.48, r - 0.46), 0.96, 0.92,
                        fill=False, edgecolor='#d62728', linewidth=2.0, zorder=4
                    ))
        ax.axhline(n_rows - 1.5, color='black', linewidth=1.8)
        ax.set_xticks(range(n_cols))
        ax.set_xticklabels([c[1] for c in col_defs], fontsize=10, fontweight='bold')
        ax.xaxis.tick_top()
        ax.set_title(block_title, fontsize=12, fontweight='bold', pad=14)
        return im

    im_mean = draw_heatmap_block(
        ax_mean, means_mat, stds_mat,
        'Mean ± Std BLOSUM Distance to GT (red box = row best)',
        show_std_all_rows=True,
    )
    im_min = draw_heatmap_block(
        ax_min, mins_mat, mins_stds_mat,
        'Minimum Sequence BLOSUM Distance to GT (red box = row best)',
        show_std_all_rows=False,
    )

    ax_mean.set_yticks(range(n_rows))
    ax_mean.set_yticklabels(row_labels, fontsize=9.5)
    for tick_idx, lbl in enumerate(ax_mean.get_yticklabels()):
        if tick_idx == n_rows - 1:
            lbl.set_fontweight('bold')
        elif row_labels[tick_idx].startswith('    ↳'):
            lbl.set_fontstyle('italic')
    ax_min.tick_params(axis='y', left=False)

    fig.suptitle(
        f'Summary of BLOSUM Distances to Ground Truth ({title_suffix})', 
        fontsize=15, fontweight='bold', y=0.995
    )
    plt.tight_layout(rect=[0, 0, 0.93, 0.965])

    # Add vertical separator line between the two table blocks
    pos_left = ax_mean.get_position()
    pos_right = ax_min.get_position()
    sep_x = 0.5 * (pos_left.x1 + pos_right.x0)
    fig.add_artist(Line2D(
        [sep_x, sep_x], [pos_left.y0, pos_left.y1 + 0.04],
        transform=fig.transFigure, color='#333333', linewidth=2.2, linestyle='--'
    ))

    cbar_ax = fig.add_axes([0.945, pos_right.y0, 0.015, pos_right.height])
    cbar = fig.colorbar(im_min, cax=cbar_ax)
    cbar.set_label('BLOSUM distance to GT', fontsize=10)

    plt.show()

plot_mean_std_and_min_summary_table(pw_distances_by_group, title_suffix='Full S1')
if 'pw_distances_by_group_rbd' in globals():
    plot_mean_std_and_min_summary_table(pw_distances_by_group_rbd, title_suffix='RBD Only')